In [1]:
import re
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://aws.amazon.com/rds/aurora/?nc2=h_prod_db_aa&trk=4f4d614b-c533-483d-b76b-9d919387c02e&sc_channel=ps")
contents = loader.load()


In [2]:
import json
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

class SafeFAISSStore:
    def __init__(self, embedding):
        self.embedding = embedding
        self.db = None

    def build(self, documents):
        """Build FAISS index from a list of Document objects."""
        self.db = FAISS.from_documents(documents, self.embedding)

    def save(self, path="faiss_store.json"):
        """Save embeddings + metadata to JSON safely."""
        if not self.db:
            raise ValueError("FAISS index not built yet.")

        # Extract vectors + metadata
        vector_data = []
        for doc in self.db.docstore._dict.values():
            vec = self.embedding.embed_query(doc.page_content)
            vector_data.append({
                "vector": vec,
                "metadata": doc.metadata,
                "text": doc.page_content
            })

        with open(path, "w") as f:
            json.dump(vector_data, f)

    def load(self, path="faiss_store.json"):
        """Load FAISS index from JSON safely (no pickle)."""
        with open(path) as f:
            data = json.load(f)

        # Rebuild FAISS index
        documents = [Document(page_content=item["text"], metadata=item["metadata"]) for item in data]
        self.db = FAISS.from_documents(documents, self.embedding)

    def query(self, query, k=3):
        """Run similarity search on the FAISS index."""
        if not self.db:
            raise ValueError("FAISS index not loaded or built yet.")
        return self.db.similarity_search(query, k=k)


In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

# Step 1: Initialize
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
store = SafeFAISSStore(embedding)

# Step 2: Build from documents
store.build(contents)

# Step 3: Save safely
store.save("aurora_store.json")

# Step 4: Load later
store.load("aurora_store.json")

# Step 5: Query
results = store.query("What is Aurora?", k=2)
for r in results:
    collapsed = re.sub(r'\n+', '\n', r.page_content)
    print(collapsed)



Relational Database – Amazon Aurora MySQL PostgreSQL – AWS
Skip to main content
     
Filter: All
 
English
Contact us
AWS Marketplace
Support  
My account  
     
Search
Filter: All
 
Sign in to console
Create account
Amazon Aurora
Overview
Engines
Features
Pricing
Resources
More
Databases
Amazon RDS
Amazon Aurora
Amazon Aurora
Unparalleled high performance and availability at global scale for PostgreSQL, MySQL, and DSQL
Get started with Aurora
Connect with an Aurora specialist
What is Aurora
Aurora has 5x the throughput of MySQL and 3x of PostgreSQL with full PostgreSQL and MySQL compatibility. Aurora also offers DSQL, the fastest distributed SQL database that is PostgreSQL-compatible. Aurora is designed for up to 99.999% multi-Region availability. With Aurora DSQL, Aurora provides virtually unlimited scale in and across regions with no infrastructure management. Aurora offers broad compliance standards and enterprise security capabilities, and support for globally distributed apps.